In [4]:
import pandas as pd
import numpy as np

import random
from faker import Faker
from ydata_profiling import ProfileReport

# 소매 데이터
    - 매장 점포 데이터
    - 상품 데이터
    - 고객 데이터

In [ ]:


fake = Faker("ko_KR")
random.seed(42)

# 5개 매장만 생성
store_data = []
royalty_types = ["고정", "매출비례"]

for i in range(1, 6):
    매장코드 = f"S{str(i).zfill(3)}"
    매장명 = f"{fake.city_name()}점"
    지역 = fake.province()   # 시/도
    주소 = fake.address().split("\n")[0]  # 한 줄 주소
    개점일자 = fake.date_between(start_date="-10y", end_date="today")
    면적 = random.randint(500, 1500)  # ㎡
    직원수 = random.randint(10, 50)
    매장등급 = random.choice(["프리미엄", "일반", "소형"])

    # 직영 여부 및 로열티 조건
    직영여부 = random.choice(["Y", "N"])
    if 직영여부 == "Y":
        로열티방식 = "없음"
        로열티조건 = None
        계약만기일 = None
    else:
        로열티방식 = random.choice(royalty_types)
        if 로열티방식 == "고정":
            로열티조건 = f"{random.randint(300, 1000) * 10000}원/월"
        else:
            로열티조건 = f"매출의 {random.choice([3, 5, 7])}%"
        계약만기일 = fake.date_between(start_date="today", end_date="+5y")

    store_data.append([
        매장코드, 매장명, 지역, 주소, 개점일자, 면적, 직원수,
        매장등급, 직영여부, 로열티방식, 로열티조건, 계약만기일
    ])

# DataFrame 생성
columns = ["매장코드", "매장명", "지역", "주소", "개점일자", "면적(㎡)", "직원수", 
           "매장등급", "직영여부", "로열티방식", "로열티조건", "계약만기일"]

stores_df = pd.DataFrame(store_data, columns=columns)

print(stores_df)


In [ ]:


fake = Faker('ko_KR')
np.random.seed(42)
random.seed(42)

# -----------------------------
# 1. 기본 설정
# -----------------------------
n_rows = 5000

# 날짜 (2023-01 ~ 2025-12)
date_range = pd.date_range("2023-01-01", "2025-12-31", freq="D")
dates = np.random.choice(date_range, n_rows)

# 매장 설정 (5개, 규모 다르게)
stores = [
    {"매장명": "티니핑월드 서울점", "매장코드": "S001", "규모": 1.5},
    {"매장명": "티니핑월드 부산점", "매장코드": "S002", "규모": 1.2},
    {"매장명": "티니핑월드 대구점", "매장코드": "S003", "규모": 1.0},
    {"매장명": "티니핑월드 광주점", "매장코드": "S004", "규모": 0.8},
    {"매장명": "티니핑월드 대전점", "매장코드": "S005", "규모": 0.6},
]
store_choices = np.random.choice(stores, n_rows, p=[0.25,0.25,0.2,0.15,0.15])

# -----------------------------
# 2. 고객 설정
# -----------------------------
n_customers = 100
customers = []
for i in range(n_customers):
    gender = random.choice(["M","F"])
    age = np.random.randint(5, 70)  # 티니핑 세계관이라 어린 나이도 허용
    customers.append({
        "고객코드": f"C{str(i+1).zfill(3)}",
        "성별": gender,
        "나이": age
    })

# 고객별 거래 특성
customer_behavior = {}
for c in customers:
    freq = np.random.choice(["high","medium","low"], p=[0.3,0.5,0.2])
    active_period = np.random.choice(["early","mid","late"], p=[0.3,0.4,0.3])
    amount_level = np.random.choice(["high","low"], p=[0.3,0.7])
    product_variety = np.random.choice(["wide","narrow"], p=[0.4,0.6])
    customer_behavior[c["고객코드"]] = {
        "freq": freq, 
        "active_period": active_period, 
        "amount_level": amount_level, 
        "product_variety": product_variety
    }

customer_choices = np.random.choice(customers, n_rows)

# -----------------------------
# 3. 티니핑 스타일 상품 생성기
# -----------------------------
adjectives = ["반짝", "꿈꾸는", "별빛", "달콤", "용기", "마법", "해피", "포근"]
characters = ["하트핑", "드림핑", "스위티핑", "버블핑", "러블리핑", "스타핑", "큐티핑", "조이핑"]
categories = ["라면", "쿠키", "주스", "빵", "아이스크림", "초콜릿", "사탕", "우유", "쥬얼리", "토이"]

products = []
for i in range(20):
    adj = random.choice(adjectives)
    char = random.choice(characters)
    cat = random.choice(categories)
    name = f"{adj}{char} {cat}"
    price = np.random.randint(1000, 50000)
    products.append({
        "상품코드": f"P{str(i+1).zfill(3)}",
        "상품명": name,
        "판매단가": price
    })

product_choices = np.random.choice(products, n_rows)

# -----------------------------
# 4. 구매수량 및 할인율
# -----------------------------
quantities = []
discounts = []
for i in range(n_rows):
    store = store_choices[i]
    cust = customer_choices[i]
    
    base_qty = np.random.poisson(2)
    if customer_behavior[cust["고객코드"]]["freq"] == "high":
        base_qty += np.random.poisson(2)
    qty = max(1, int(base_qty * store["규모"]))
    quantities.append(qty)
    
    base_discount = np.random.uniform(0,0.1)
    if customer_behavior[cust["고객코드"]]["amount_level"] == "high":
        base_discount += np.random.uniform(0,0.05)
    discounts.append(round(min(base_discount,0.15),2))

# -----------------------------
# 5. 데이터프레임 생성
# -----------------------------
df = pd.DataFrame({
    "날짜": dates,
    "매장명": [s["매장명"] for s in store_choices],
    "매장코드": [s["매장코드"] for s in store_choices],
    "고객코드": [c["고객코드"] for c in customer_choices],
    "성별": [c["성별"] for c in customer_choices],
    "나이": [c["나이"] for c in customer_choices],
    "상품코드": [p["상품코드"] for p in product_choices],
    "상품명": [p["상품명"] for p in product_choices],
    "판매단가": [p["판매단가"] for p in product_choices],
    "구매수량": quantities,
    "할인율": discounts
})

df.head()


,날짜,매장명,매장코드,고객코드,성별,나이,상품코드,상품명,판매단가,구매수량,할인율
0,2025-05-10,티니핑월드 부산점,S002,C056,F,11,P007,달콤스위티핑 초콜릿,8101,3,0.07
1,2025-12-31,티니핑월드 대구점,S003,C030,M,13,P006,별빛드림핑 아이스크림,14509,3,0.08
2,2025-11-10,티니핑월드 대구점,S003,C022,M,38,P011,달콤하트핑 빵,37180,5,0.07
3,2023-05-02,티니핑월드 대전점,S005,C075,M,6,P008,별빛하트핑 토이,5743,3,0.05
4,2024-04-11,티니핑월드 대전점,S005,C081,F,17,P020,반짝버블핑 토이,37881,1,0.05


In [2]:
df.to_csv('sales_data.csv')

In [5]:
df = pd.read_csv('sales_data.csv')

In [6]:
df

,Unnamed: 0,날짜,매장명,매장코드,고객코드,성별,나이,상품코드,상품명,판매단가,구매수량,할인율
0,0,2025-05-10,티니핑월드 부산점,S002,C056,F,11,P007,달콤스위티핑 초콜릿,8101,3,0.07
1,1,2025-12-31,티니핑월드 대구점,S003,C030,M,13,P006,별빛드림핑 아이스크림,14509,3,0.08
2,2,2025-11-10,티니핑월드 대구점,S003,C022,M,38,P011,달콤하트핑 빵,37180,5,0.07
3,3,2023-05-02,티니핑월드 대전점,S005,C075,M,6,P008,별빛하트핑 토이,5743,3,0.05
4,4,2024-04-11,티니핑월드 대전점,S005,C081,F,17,P020,반짝버블핑 토이,37881,1,0.05
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4995,2023-01-13,티니핑월드 광주점,S004,C022,M,38,P018,포근드림핑 빵,17543,1,0.08
4996,4996,2025-11-07,티니핑월드 부산점,S002,C030,M,13,P002,용기하트핑 쿠키,18705,2,0.04
4997,4997,2024-11-24,티니핑월드 서울점,S001,C062,M,31,P001,해피조이핑 쥬얼리,11961,6,0.06
4998,4998,2025-08-30,티니핑월드 대전점,S005,C073,F,20,P016,달콤러블리핑 사탕,3470,1,0.10


In [8]:

report = ProfileReport(df, title="티니핑 소매점 데이터 리포트", explorative=True)
# report.to_notebook_iframe()  # 주피터 노트북에서 보기
report.to_file("report.html")  # HTML로 저장

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 12/12 [00:00<00:00, 420.62it/s]
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 45208 (\N{HANGUL SYLLABLE NA}) missing from current font.
  plt.savefig(
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 51060 (\N{HANGUL SYLLABLE I}) missing from current font.
  plt.savefig(
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 54032 (\N{HANGUL SYLLABLE PAN}) missing from current font.
  plt.savefig(
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 47588 (\N{HANGUL SYLLABLE MAE}) missing from current font.
  plt.savefig(
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 45800 (\N{HANGUL SYLLABLE DAN}) missing from current font.
  plt.savefig(

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 54000 (\N{HANGUL SYLLABLE TI}) missing from current font.
  plt.savefig(
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 45768 (\N{HANGUL SYLLABLE NI}) missing from current font.
  plt.savefig(
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 54609 (\N{HANGUL SYLLABLE PING}) missing from current font.
  plt.savefig(
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 50900 (\N{HANGUL SYLLABLE WEOL}) missing from current font.
  plt.savefig(
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site-packages\ydata_profiling\visualisation\utils.py:73: UserWarning: Glyph 46300 (\N{HANGUL SYLLABLE DEU}) missing from current font.
  plt.savefig(
c:\Users\pooko\anaconda3\envs\bibiml\Lib\site

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]